In [2]:
import pandas as pd
movies=pd.read_csv("movies_cleaned.csv")
cast=pd.read_csv("cast_cleaned.csv")
crew=pd.read_csv("crew_cleaned.csv")
genres=pd.read_csv("genres_cleaned.csv")
movie_genres=pd.read_csv("movie_genres_cleaned.csv")
movie_keywords=pd.read_csv("movie_keywords_cleaned.csv")


In [3]:


total_movies = movies["movie_id"].nunique()
total_genres = genres["genre_id"].nunique()
total_cast_members = cast["person_id"].nunique()
total_crew_members = crew["person_id"].nunique()
total_movies
total_genres
total_cast_members
total_crew_members


2848

In [5]:

problem_movies = movies[
    movies["budget"].isna() |
    (movies["budget"] == 0) |
    movies["revenue"].isna() |
    (movies["revenue"] == 0)
]

problem_movies[
    [ "title", "budget", "revenue"]
]


,title,budget,revenue
324,Zack Snyder's Justice League,70000000.0,0.0
340,Bird Box,19800000.0,0.0
403,365 Days,0.0,9458590.0
468,To All the Boys I've Loved Before,0.0,0.0
554,Prey,65000000.0,0.0
...,...,...,...
2469,Day Shift,100000000.0,0.0
2471,A Serbian Film,0.0,0.0
2480,Chungking Express,160000.0,0.0
2481,I'm Thinking of Ending Things,0.0,0.0


In [29]:

valid_movies["profit"] = valid_movies["revenue"] - valid_movies["budget"]
valid_movies["ROI"] = (valid_movies["profit"] / valid_movies["budget"]) * 100
valid_movies[["title", "profit", "ROI"]]


C:\Users\dines\AppData\Local\Temp\ipykernel_20992\297879013.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_movies["profit"] = valid_movies["revenue"] - valid_movies["budget"]
C:\Users\dines\AppData\Local\Temp\ipykernel_20992\297879013.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_movies["ROI"] = (valid_movies["profit"] / valid_movies["budget"]) * 100


,title,profit,ROI
0,Interstellar,5.816067e+08,352.488913
1,Inception,6.790306e+08,424.394144
2,The Avengers,1.298816e+09,590.370689
3,The Dark Knight,8.195584e+08,443.004564
4,Avatar,2.686706e+09,1133.631235
...,...,...,...
2498,Daybreakers,3.141719e+07,157.085940
2499,Date Night,-5.619119e+06,-6.028140
2500,Transformers,1.646479e+08,149.545660
2501,The SpongeBob Movie: Sponge Out of Water,5.313726e+08,1352.613653


In [30]:
movies["ROI"] = ((movies["revenue"] - movies["budget"]) / movies["budget"]) * 100

movies[movies["budget"] >= 1000000].sort_values(
    by="ROI", ascending=False
).head(15)[["title", "budget", "revenue", "ROI"]]

,title,budget,revenue,ROI
1767,Dragon Ball Super: Broly,1000000.0,125002821.0,12400.282100
535,Snow White and the Seven Dwarfs,1488423.0,184925486.0,12324.256142
1791,The Rocky Horror Picture Show,1400000.0,171181400.0,12127.242857
467,Rocky,1000000.0,117253345.0,11625.334500
1227,Gone with the Wind,4000000.0,402352579.0,9958.814475
716,The Jungle Book,4000000.0,378000000.0,9350.000000
635,Cinderella,2900000.0,263600000.0,8989.655172
372,Saw,1200000.0,104045735.0,8570.477917
719,One Hundred and One Dalmatians,3600000.0,303000000.0,8316.666667
263,E.T. the Extra-Terrestrial,10500000.0,797307407.0,7493.403876


In [32]:
movies_per_year = movies.groupby("release_year")["movie_id"].count()
movies_per_year
movies_per_year.idxmax()
movies_per_year.max()

121

In [34]:
Q1 = movies["runtime"].quantile(0.25)
Q3 = movies["runtime"].quantile(0.75)
IQR = Q3 - Q1
outliers = movies[
    (movies["runtime"] < Q1 - 1.5 * IQR) |
    (movies["runtime"] > Q3 + 1.5 * IQR)
]
outliers[["title", "runtime"]]

,title,runtime
0,Interstellar,169
16,Avengers: Endgame,181
18,The Lord of the Rings: The Fellowship of the Ring,179
19,Titanic,194
20,The Lord of the Rings: The Return of the King,201
21,The Wolf of Wall Street,180
28,The Lord of the Rings: The Two Towers,179
38,The Godfather,175
76,The Hobbit: An Unexpected Journey,169
79,The Green Mile,189


In [ ]:
primary_genre = movie_genres.drop_duplicates("movie_id")
primary_genre = pd.merge(primary_genre, genres, on="genre_id")
primary_genre = pd.merge(primary_genre, movies, on="movie_id")
average_popularity = primary_genre.groupby("genre_name")["popularity"].mean()
average_popularity.sort_values(ascending=False)

genre_name
Science Fiction    21.261836
Music              18.324340
Animation          16.811212
Family             16.710635
Action             16.385188
War                15.470317
Horror             14.851835
Adventure          14.785510
Fantasy            13.022920
Mystery            12.427131
Western            12.322680
Thriller           11.294211
Drama              11.278534
Crime              10.739498
History            10.414567
Comedy              9.984262
Romance             9.976708
Documentary         7.726200
Name: popularity, dtype: float64

In [ ]:
actor_ratings = cast.merge(movies, on="movie_id")
actor_ratings = actor_ratings.groupby("actor_name").agg({
    "movie_id": "nunique",
    "vote_average": "mean"
})
actor_ratings.sort_values("movie_id", ascending=False).head(10)

,movie_id,vote_average
actor_name,,
Samuel L. Jackson,39,6.938692
Brad Pitt,38,7.257289
Robert De Niro,38,7.228316
Johnny Depp,37,6.856973
Tom Hanks,33,7.289758
Willem Dafoe,32,7.114781
Mark Wahlberg,32,6.630063
Tom Cruise,31,7.040613
Scarlett Johansson,31,7.087194


In [37]:
duplicate_movies = movies[
    movies.duplicated(subset="title", keep=False)
].sort_values("title")
duplicate_movies[["title", "release_year"]]

,title,release_year
878,A Nightmare on Elm Street,1984
1928,A Nightmare on Elm Street,2010
261,Aladdin,1992
339,Aladdin,2019
792,Alice in Wonderland,1951
...,...,...
1737,The Thing,2011
860,Total Recall,2012
744,Total Recall,1990
252,Transformers,2007


In [39]:
unusual_movies = movies[
    ((movies["popularity"] > movies["popularity"].mean()) &
     (movies["vote_count"] < movies["vote_count"].mean())) |
    ((movies["popularity"] < movies["popularity"].mean()) &
     (movies["vote_count"] > movies["vote_count"].mean()))
]

unusual_movies[["title", "popularity", "vote_count"]]

,title,popularity,vote_count
34,The Hunger Games,12.2061,23506
70,It,13.4174,20496
101,The Silence of the Lambs,7.0584,18235
104,Bohemian Rhapsody,12.4770,17898
115,Lucy,13.5829,17219
...,...,...,...
2452,Mortal Kombat II,99.5231,2260
2457,Smile 2,23.7112,2256
2471,A Serbian Film,18.9190,2242
2493,Lilo & Stitch,33.2161,2228
